In [1]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import glob
import os
import gc
import pyarrow
import pyarrow.parquet
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from datetime import timedelta
from pandas import ExcelWriter
from google.cloud import bigquery
from itertools import combinations, permutations

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth",0)
pd.options.display.float_format = '{:,}'.format
import calendar

from dateutil.relativedelta import relativedelta
import io


from google.cloud import bigquery
from google.cloud import storage
#import pyodbc
import re



import pandas as pd
from sklearn.preprocessing import OneHotEncoder

import pandas as pd
from datetime import timedelta

from sklearn.feature_extraction.text import TfidfVectorizer


import traceback
import string
from pandas.tseries.offsets import DateOffset
import os
from google.cloud import bigquery
from google.cloud import storage
#import pyodbc
import re
import pandas as pd
import numpy as np
import pandas.core.algorithms as algos
from pandas import Series




from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier,Pool
import optuna
from sklearn.model_selection import train_test_split,GridSearchCV,cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve,confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import precision_recall_curve
import optuna
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV

import seaborn as sns
import matplotlib.pyplot as plt
import pickle
from scipy.stats import chi2_contingency

In [2]:
%%bigquery planet_portfolio_penetration


WITH DIM_AGG AS (
    SELECT 
        b.AGREEMENTNO,
        a.CUSTOMERID,
        a.DOB,
        a.MOBILE,
        a.CITY,
        b.SOURCESYSTEMID,
        b.LOANSTATUS,
        b.STATE,
        b.DISBURSALDATE,
        b.DISBURSALAMOUNT,
        b.SCHEMENAME,
        b.PRODUCTNAME,
        a.UCID,
        b.PRODUCTID,
        b.NPASTAGEID
    FROM `edw-prod.DWH.LMS_DIMCUSTOMER` a
    LEFT JOIN (
        SELECT AGREEMENTNO, CUSTOMERID, SOURCESYSTEMID, LOANSTATUS, DISBURSALDATE, DISBURSALAMOUNT, SCHEMENAME, PRODUCTNAME, PRODUCTID, STATE, NPASTAGEID 
        FROM `edw-prod.DWH.LMSFR_DIMAGREEMENT`
        UNION ALL
        SELECT AGREEMENTNO, CUSTOMERID, SOURCESYSTEMID, LOANSTATUS, DISBURSALDATE, DISBURSALAMOUNT, SCHEMENAME, PRODUCTNAME, PRODUCTID, STATE, NPASTAGEID 
        FROM `edw-prod.DWH.LMS_DIMAGREEMENT`
        UNION ALL
        SELECT AGREEMENTNO, CUSTOMERID, SOURCESYSTEMID, LOANSTATUS, DISBURSALDATE, DISBURSALAMOUNT, SCHEMENAME, PRODUCTNAME, PRODUCTID, STATE, NPASTAGEID 
        FROM `edw-prod.DWH.LMSML_DIMAGREEMENT`
    ) b
    ON a.CUSTOMERID = b.CUSTOMERID
    WHERE b.AGREEMENTNO IS NOT NULL
    AND b.LOANSTATUS IN ('A', 'C', 'ACTIVE', 'CLOSED')
),

DIM_AGG_FINAL AS (
    SELECT *,
        CASE 
            WHEN SOURCESYSTEMID = 21 THEN "TW"
            WHEN SOURCESYSTEMID = 20 THEN "HL"
            WHEN SOURCESYSTEMID = 23 THEN "CL"
            WHEN SOURCESYSTEMID = 25 THEN "SME"
            WHEN SOURCESYSTEMID = 24 THEN "Farm"
            WHEN SOURCESYSTEMID = 27 THEN "WRF"
            WHEN SOURCESYSTEMID = 15 THEN "ML"
            WHEN SOURCESYSTEMID = 1 THEN "Farm"
            WHEN SOURCESYSTEMID = 6 THEN "Wholesale"
            WHEN SOURCESYSTEMID = 26 THEN "Rural-LAP"
            ELSE "Non-LTF"
        END AS PRODUCT,
        CASE 
            WHEN SCHEMENAME = "SUFIN" THEN "Supply-Chain-Finance"
            WHEN LOWER(PRODUCTNAME) LIKE '%lap%' OR LOWER(PRODUCTNAME) LIKE '%loan against property%' AND SOURCESYSTEMID = 20 THEN "Loan-Against-Property"
            ELSE CASE 
                WHEN SOURCESYSTEMID = 21 THEN "TW"
                WHEN SOURCESYSTEMID = 20 THEN "HL"
                WHEN SOURCESYSTEMID = 23 THEN "CL"
                WHEN SOURCESYSTEMID = 25 THEN "SME"
                WHEN SOURCESYSTEMID = 24 THEN "Farm"
                WHEN SOURCESYSTEMID = 27 THEN "WRF"
                WHEN SOURCESYSTEMID = 15 THEN "ML"
                WHEN SOURCESYSTEMID = 1 THEN "Farm"
                WHEN SOURCESYSTEMID = 6 THEN "Wholesale"
                WHEN SOURCESYSTEMID = 26 THEN "Rural-LAP"
                ELSE "Non-LTF"
            END
        END AS Product_Final
    FROM DIM_AGG
),

DIM_AGG_CHECK AS (
    SELECT * FROM DIM_AGG_FINAL
    WHERE MOBILE IS NOT NULL AND MOBILE != 'None'
),

DIM_AGG_CHECK_FINAL AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY UCID, Product_Final ORDER BY DISBURSALDATE DESC) AS rn
        FROM DIM_AGG_CHECK
        WHERE UCID IS NOT NULL
    ) WHERE rn = 1
),

PRODUCT_DIST_FINAL AS (
    SELECT 
        Product_Final, 
        CASE 
            WHEN Product_Final IN ('ML','TW') AND NPASTAGEID = 'WRITEOFF' THEN 'WRITEOFF'
            WHEN Product_Final IN ('ML','TW') THEN LOANSTATUS
            ELSE LOANSTATUS 
        END AS Final_Status, 
        COUNT(DISTINCT UCID) AS unique_agreement_count,
        'PRODUCT' AS Source_Tag  
    FROM DIM_AGG_CHECK_FINAL
    GROUP BY Product_Final, Final_Status
),

PLANET_REG AS (
    SELECT USER_PHNO1 AS MOBILE, USER_ID, CREATE_TS AS Planet_Reg_Date
  FROM `dataeng-datalake-prod.D2C.TB_ASMI_USER`
  UNION ALL
  SELECT USER_PHNO1 AS MOBILE, USER_ID,CREATE_TS AS Planet_Reg_Date
  FROM `edw-prod.DWH_MART.D2C_TB_ASMI_USER`
),

PLANET_AGG AS (
    SELECT 
        p.MOBILE AS MOBILE_ASMI, 
        p.USER_ID, 
        p.Planet_Reg_Date, 
        d.AGREEMENTNO, 
        d.MOBILE, 
        d.UCID, 
        d.STATE, 
        d.LOANSTATUS, 
        d.PRODUCT, 
        d.Product_Final, 
        d.DISBURSALDATE, 
        d.NPASTAGEID
    FROM PLANET_REG p
    LEFT JOIN DIM_AGG_FINAL d
    ON p.MOBILE = d.MOBILE
),

PLANET_CHECK_FINAL AS (
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY UCID, Product_Final ORDER BY DISBURSALDATE DESC) AS rn
        FROM PLANET_AGG
        WHERE UCID IS NOT NULL
    ) WHERE rn = 1
),

PLANET_DIST_FINAL AS (
    SELECT 
        Product_Final, 
        CASE 
            WHEN Product_Final IN ('ML','TW') AND NPASTAGEID = 'WRITEOFF' THEN 'WRITEOFF'
            WHEN Product_Final IN ('ML','TW') THEN LOANSTATUS
            ELSE LOANSTATUS 
        END AS Final_Status, 
        COUNT(DISTINCT UCID) AS unique_ucid_count,
        'PLANET' AS Source_Tag  
    FROM PLANET_CHECK_FINAL
    GROUP BY Product_Final, Final_Status
)

SELECT * FROM PRODUCT_DIST_FINAL
UNION ALL
SELECT * FROM PLANET_DIST_FINAL;


Query is running:   0%|          |

Downloading:   0%|          |

In [4]:
planet_portfolio_penetration.pivot(
    index=[ 'Product_Final'],
    columns=['Source_Tag','Final_Status'],
    values='unique_agreement_count'
).fillna(0)

Source_Tag             PRODUCT                     PLANET                  
Final_Status                 C WRITEOFF        A        C        A WRITEOFF
Product_Final                                                              
CL                     470521   0        773476   330387   584560   0      
Farm                   660145   0        463798   102225   172789   0      
HL                     18911    0        45220    13509    37566    0      
Loan-Against-Property  5484     0        11809    3650     9304     0      
ML                     5630920  2997623  5614873  686489   813861   133252 
Non-LTF                0        0        28       0        8        0      
Rural-LAP              489      0        17547    277      5261     0      
SME                    6379     0        46202    5037     33776    0      
TW                     6103967  496417   2333552  2227825  1725389  91241  
WRF                    339      0        193      230      104      0

# MAU

In [2]:
%%bigquery MAU_lob
WITH user_mapping AS (
  SELECT PHONE_NO AS planet_phone_no, USER_CREATION_ID AS USER_ID
  FROM `dataeng-datalake-prod.D2C.D2C_USER_REG_LOGIN`
  UNION ALL
  SELECT USER_PHNO1 AS planet_phone_no, USER_ID
  FROM `edw-prod.DWH_MART.D2C_TB_ASMI_USER`
),

loc_with_mobile AS (
  -- All login events with mobile mapping
  SELECT 
    l.USER_ID,
    l.CREATED_TS,
    m.planet_phone_no
  FROM (
    SELECT USER_ID, CREATED_TS
    FROM `edw-prod.DWH_MART.D2C_TB_USER_LOC_TRACKING`
    WHERE CREATED_TS >= '2025-01-01'
      AND CREATED_TS < '2026-06-01'
    
    UNION ALL
    
    SELECT USER_CREATION_ID AS USER_ID, LOGIN_DATE AS CREATED_TS
    FROM `dataeng-datalake-prod.D2C.D2C_USER_LOGIN_TRACKER`
    WHERE LOGIN_DATE >= '2025-01-01'
      AND LOGIN_DATE < '2026-06-01'
  ) l
  LEFT JOIN user_mapping m
    ON l.USER_ID = m.USER_ID
),

dim_agg AS (
  SELECT 
    b.AGREEMENTNO,
    a.CUSTOMERID,
    a.DOB,
    a.MOBILE,
    b.SOURCESYSTEMID,
    b.LOANSTATUS,
    b.SCHEMENAME,
    b.PRODUCTNAME,
    a.UCID,
    b.PRODUCTID
  FROM (
    SELECT CUSTOMERID, DOB, MOBILE, UCID
    FROM `edw-prod.DWH.LMS_DIMCUSTOMER`
  ) a
  LEFT JOIN (
    SELECT AGREEMENTNO, CUSTOMERID, SOURCESYSTEMID, LOANSTATUS, SCHEMENAME, PRODUCTNAME, PRODUCTID 
    FROM `edw-prod.DWH.LMSFR_DIMAGREEMENT`
    UNION ALL
    SELECT AGREEMENTNO, CUSTOMERID, SOURCESYSTEMID, LOANSTATUS, SCHEMENAME, PRODUCTNAME, PRODUCTID 
    FROM `edw-prod.DWH.LMS_DIMAGREEMENT`
    UNION ALL
    SELECT AGREEMENTNO, CUSTOMERID, SOURCESYSTEMID, LOANSTATUS, SCHEMENAME, PRODUCTNAME, PRODUCTID 
    FROM `edw-prod.DWH.LMSML_DIMAGREEMENT`
  ) b
  ON a.CUSTOMERID = b.CUSTOMERID 
  WHERE b.AGREEMENTNO IS NOT NULL
    AND b.LOANSTATUS IN ('A','C','ACTIVE','CLOSED')
),

dim_agg_final AS (
  SELECT * ,
    CASE 
        WHEN SOURCESYSTEMID = 21 THEN "TW"
        WHEN SOURCESYSTEMID = 20 THEN "HL"
        WHEN SOURCESYSTEMID = 23 THEN "CL"
        WHEN SOURCESYSTEMID = 25 THEN "SME"
        WHEN SOURCESYSTEMID = 24 THEN "Farm"
        WHEN SOURCESYSTEMID = 27 THEN "WRF"
        WHEN SOURCESYSTEMID = 15 THEN "ML"
        WHEN SOURCESYSTEMID = 1 THEN "Farm"
        WHEN SOURCESYSTEMID = 6 THEN "Wholesale"
        WHEN SOURCESYSTEMID = 26 THEN "Rural-LAP"
        ELSE "Non-LTF"
    END AS PRODUCT,
    CASE 
        WHEN SCHEMENAME = "SUFIN" THEN "Supply-Chain-Finance"
        WHEN (LOWER(PRODUCTNAME) LIKE '%lap%' OR LOWER(PRODUCTNAME) LIKE '%loan against property%') 
             AND SOURCESYSTEMID = 20 THEN "Loan-Against-Property"
        ELSE CASE 
            WHEN SOURCESYSTEMID = 21 THEN "TW"
            WHEN SOURCESYSTEMID = 20 THEN "HL"
            WHEN SOURCESYSTEMID = 23 THEN "CL"
            WHEN SOURCESYSTEMID = 25 THEN "SME"
            WHEN SOURCESYSTEMID = 24 THEN "Farm"
            WHEN SOURCESYSTEMID = 27 THEN "WRF"
            WHEN SOURCESYSTEMID = 15 THEN "ML"
            WHEN SOURCESYSTEMID = 1 THEN "Farm"
            WHEN SOURCESYSTEMID = 6 THEN "Wholesale"
            WHEN SOURCESYSTEMID = 26 THEN "Rural-LAP"
            ELSE "Non-LTF"
        END
    END AS Product_Final
  FROM dim_agg
),

final_data AS (
  -- Join login activity with product info and count MAU per product, per loanstatus, per month
  SELECT
    FORMAT_DATE('%Y-%m', DATE(l.CREATED_TS)) AS CREATED_MONTH,
    d.Product_Final,
    d.LOANSTATUS,
    COUNT(DISTINCT l.USER_ID) AS Monthly_Active_Users
  FROM loc_with_mobile l
  LEFT JOIN dim_agg_final d
    ON l.planet_phone_no = d.MOBILE
  GROUP BY CREATED_MONTH, d.Product_Final, d.LOANSTATUS
)

SELECT 
  CREATED_MONTH,
  Product_Final,
  LOANSTATUS,
  Monthly_Active_Users
FROM final_data
ORDER BY CREATED_MONTH, Product_Final, LOANSTATUS;

Query is running:   0%|          |

Downloading:   0%|          |

In [3]:
MAU_lob

,CREATED_MONTH,Product_Final,LOANSTATUS,Monthly_Active_Users
0,2025-01,None,None,345084
1,2025-01,CL,A,184682
2,2025-01,CL,C,175208
3,2025-01,Farm,A,25163
4,2025-01,Farm,C,19108
5,2025-01,HL,A,11174
6,2025-01,HL,C,4557
7,2025-01,Loan-Against-Property,A,2299
8,2025-01,Loan-Against-Property,C,910
9,2025-01,ML,A,90920


In [4]:
MAU_lob[MAU_lob['CREATED_MONTH']=="2025-08"]

,CREATED_MONTH,Product_Final,LOANSTATUS,Monthly_Active_Users
133,2025-08,None,None,241497
134,2025-08,CL,A,177315
135,2025-08,CL,C,133085
136,2025-08,Farm,A,30212
137,2025-08,Farm,C,17433
138,2025-08,HL,A,10887
139,2025-08,HL,C,3609
140,2025-08,Loan-Against-Property,A,2687
141,2025-08,Loan-Against-Property,C,759
142,2025-08,ML,A,93072


In [5]:
MAU_lob['Product_Final'] = MAU_lob['Product_Final'].fillna("New Customer")

In [6]:
MAU_lob_agg = (
    MAU_lob.groupby(["CREATED_MONTH", "Product_Final"], as_index=False)
           .agg({"Monthly_Active_Users": "sum"})
)

pivot_df = MAU_lob_agg.pivot(
    index="CREATED_MONTH",
    columns="Product_Final",
    values="Monthly_Active_Users",
).reset_index()

pivot_df


Product_Final,CREATED_MONTH,CL,Farm,HL,Loan-Against-Property,ML,New Customer,Non-LTF,Rural-LAP,SME,TW,WRF
0,2025-01,359890,44271,15731,3209,220344,345084,<NA>,503,9785,758398,54
1,2025-02,340508,37332,15361,3181,204120,247151,<NA>,467,9852,736185,53
2,2025-03,347801,36078,15196,3308,193190,261759,<NA>,539,10570,787923,56
3,2025-04,337729,34478,16628,3409,188326,240923,<NA>,602,10771,771827,63
4,2025-05,342765,40220,16431,3565,188715,231543,<NA>,732,12272,790329,78
5,2025-06,348028,44265,20482,4106,198554,240117,<NA>,911,12036,821271,65
6,2025-07,355778,44867,18563,3926,211123,247566,<NA>,1050,11898,839995,85
7,2025-08,310400,47645,14496,3446,197186,241497,<NA>,1034,10631,761498,89
8,2025-09,280589,43972,14973,3542,187226,252505,<NA>,1103,11339,736587,74
9,2025-10,282218,40945,12619,3247,182529,265900,<NA>,1248,11612,814556,80


In [7]:
MAU_lob_apr26 = MAU_lob[(MAU_lob['CREATED_MONTH']=="2026-04")|(MAU_lob['CREATED_MONTH']=="2026-05")]
MAU_lob_agg = (
    MAU_lob_apr26.groupby(["CREATED_MONTH", "Product_Final"], as_index=False)
           .agg({"Monthly_Active_Users": "sum"})
)

pivot_df = MAU_lob_agg.pivot(
    index="CREATED_MONTH",
    columns="Product_Final",
    values="Monthly_Active_Users",
).reset_index()

pivot_df


Product_Final,CREATED_MONTH,CL,Farm,HL,Loan-Against-Property,ML,New Customer,Non-LTF,Rural-LAP,SME,TW,WRF
0,2026-04,257043,38755,13567,3653,177591,217123,2,1730,14299,744633,111
1,2026-05,183982,26957,8786,2454,147685,99026,<NA>,1371,8854,523174,41


In [1]:
MAU_lob.pivot(
    index=['CREATED_MONTH',],
    columns=['Product_Final','LOANSTATUS'],
    values='Monthly_Active_Users'
)

NameError: name 'MAU_lob' is not defined

# New Customer

In [2]:
%%bigquery new_customer
WITH user_mapping AS (
  SELECT PHONE_NO AS planet_phone_no, USER_CREATION_ID AS USER_ID
  FROM `dataeng-datalake-prod.D2C.D2C_USER_REG_LOGIN`
  UNION ALL
  SELECT USER_PHNO1 AS planet_phone_no, USER_ID
  FROM `edw-prod.DWH_MART.D2C_TB_ASMI_USER`
),

dim_agg AS (
  SELECT 
    b.AGREEMENTNO,
    a.CUSTOMERID,
    a.DOB,
    a.MOBILE,
    b.SOURCESYSTEMID,
    b.LOANSTATUS,
    b.SCHEMENAME,
    b.PRODUCTNAME,
    a.UCID,
    b.PRODUCTID
  FROM (
    SELECT CUSTOMERID, DOB, MOBILE, UCID
    FROM `edw-prod.DWH.LMS_DIMCUSTOMER`
  ) a
  LEFT JOIN (
    SELECT AGREEMENTNO, CUSTOMERID, SOURCESYSTEMID, LOANSTATUS, SCHEMENAME, PRODUCTNAME, PRODUCTID 
    FROM `edw-prod.DWH.LMSFR_DIMAGREEMENT`
    UNION ALL
    SELECT AGREEMENTNO, CUSTOMERID, SOURCESYSTEMID, LOANSTATUS, SCHEMENAME, PRODUCTNAME, PRODUCTID 
    FROM `edw-prod.DWH.LMS_DIMAGREEMENT`
    UNION ALL
    SELECT AGREEMENTNO, CUSTOMERID, SOURCESYSTEMID, LOANSTATUS, SCHEMENAME, PRODUCTNAME, PRODUCTID 
    FROM `edw-prod.DWH.LMSML_DIMAGREEMENT`
  ) b
  ON a.CUSTOMERID = b.CUSTOMERID 
  WHERE b.AGREEMENTNO IS NOT NULL
    AND b.LOANSTATUS IN ('A','C','ACTIVE','CLOSED')
),

dim_agg_final AS (
  SELECT * ,
    CASE 
        WHEN SOURCESYSTEMID = 21 THEN "TW"
        WHEN SOURCESYSTEMID = 20 THEN "HL"
        WHEN SOURCESYSTEMID = 23 THEN "CL"
        WHEN SOURCESYSTEMID = 25 THEN "SME"
        WHEN SOURCESYSTEMID = 24 THEN "Farm"
        WHEN SOURCESYSTEMID = 27 THEN "WRF"
        WHEN SOURCESYSTEMID = 15 THEN "ML"
        WHEN SOURCESYSTEMID = 1 THEN "Farm"
        WHEN SOURCESYSTEMID = 6 THEN "Wholesale"
        WHEN SOURCESYSTEMID = 26 THEN "Rural-LAP"
        ELSE "Non-LTF"
    END AS PRODUCT,
    CASE 
        WHEN SCHEMENAME = "SUFIN" THEN "Supply-Chain-Finance"
        WHEN (LOWER(PRODUCTNAME) LIKE '%lap%' OR LOWER(PRODUCTNAME) LIKE '%loan against property%') 
             AND SOURCESYSTEMID = 20 THEN "Loan-Against-Property"
        ELSE CASE 
            WHEN SOURCESYSTEMID = 21 THEN "TW"
            WHEN SOURCESYSTEMID = 20 THEN "HL"
            WHEN SOURCESYSTEMID = 23 THEN "CL"
            WHEN SOURCESYSTEMID = 25 THEN "SME"
            WHEN SOURCESYSTEMID = 24 THEN "Farm"
            WHEN SOURCESYSTEMID = 27 THEN "WRF"
            WHEN SOURCESYSTEMID = 15 THEN "ML"
            WHEN SOURCESYSTEMID = 1 THEN "Farm"
            WHEN SOURCESYSTEMID = 6 THEN "Wholesale"
            WHEN SOURCESYSTEMID = 26 THEN "Rural-LAP"
            ELSE "Non-LTF"
        END
    END AS Product_Final
  FROM dim_agg
),

-- Join Planet users to Agreement data
mapped AS (
  SELECT 
    u.planet_phone_no,
    u.USER_ID,
    d.CUSTOMERID,
    d.AGREEMENTNO,
    d.LOANSTATUS,
    d.Product_Final
  FROM user_mapping u
  LEFT JOIN dim_agg_final d
    ON SAFE_CAST(u.planet_phone_no AS STRING) = SAFE_CAST(d.MOBILE AS STRING)
)

-- Final aggregation
SELECT
  Product_Final,
  LOANSTATUS,
  COUNT(DISTINCT CUSTOMERID) AS customer_count,
  COUNT(DISTINCT CASE WHEN AGREEMENTNO IS NULL THEN USER_ID END) AS planet_only_userid_count,
  COUNT(DISTINCT CASE WHEN AGREEMENTNO IS NULL THEN planet_phone_no END) AS planet_only_phone_count
FROM mapped
GROUP BY Product_Final, LOANSTATUS
ORDER BY Product_Final, LOANSTATUS;


Query is running:   0%|          |

Downloading:   0%|          |

In [3]:
new_customer

,Product_Final,LOANSTATUS,customer_count,planet_only_userid_count,planet_only_phone_count
0,None,None,0,6392842,6380164
1,CL,A,556199,0,0
2,CL,C,419652,0,0
3,Farm,A,165613,0,0
4,Farm,C,157872,0,0
5,HL,A,48098,0,0
6,HL,C,15007,0,0
7,Loan-Against-Property,A,9596,0,0
8,Loan-Against-Property,C,3777,0,0
9,ML,A,981834,0,0


# DAU

In [5]:
%%bigquery loc_with_mobile_flagged
WITH user_mapping AS (
  SELECT 
    PHONE_NO AS planet_phone_no, 
    USER_CREATION_ID AS USER_ID
  FROM `dataeng-datalake-prod.D2C.D2C_USER_REG_LOGIN`
  
  UNION ALL
  
  SELECT 
    USER_PHNO1 AS planet_phone_no, 
    USER_ID
  FROM `edw-prod.DWH_MART.D2C_TB_ASMI_USER`
),

loc_with_mobile AS (
  -- All login events with mobile mapping
  SELECT 
    l.USER_ID,
    DATE(l.CREATED_TS) AS LOGIN_DATE,
    m.planet_phone_no
  FROM (
    SELECT USER_ID, CREATED_TS
    FROM `edw-prod.DWH_MART.D2C_TB_USER_LOC_TRACKING`
    WHERE CREATED_TS >= '2025-06-01'
      AND CREATED_TS < '2025-12-01'
    
    UNION ALL
    
    SELECT USER_CREATION_ID AS USER_ID, LOGIN_DATE AS CREATED_TS
    FROM `dataeng-datalake-prod.D2C.D2C_USER_LOGIN_TRACKER`
    WHERE LOGIN_DATE >= '2025-06-01'
      AND LOGIN_DATE < '2025-12-01'
  ) l
  LEFT JOIN user_mapping m
    ON l.USER_ID = m.USER_ID
),

dim_agg AS (
  SELECT 
    b.AGREEMENTNO,
    a.MOBILE
  FROM `edw-prod.DWH.LMS_DIMCUSTOMER` a
  LEFT JOIN (
    SELECT AGREEMENTNO, CUSTOMERID 
    FROM `edw-prod.DWH.LMSFR_DIMAGREEMENT`
    UNION ALL
    SELECT AGREEMENTNO, CUSTOMERID 
    FROM `edw-prod.DWH.LMS_DIMAGREEMENT`
    UNION ALL
    SELECT AGREEMENTNO, CUSTOMERID 
    FROM `edw-prod.DWH.LMSML_DIMAGREEMENT`
  ) b
  ON a.CUSTOMERID = b.CUSTOMERID 
  WHERE b.AGREEMENTNO IS NOT NULL
)

SELECT 
  l.USER_ID,
  l.LOGIN_DATE,
  l.planet_phone_no,
  CASE 
    WHEN u.USER_ID IS NOT NULL THEN 1 
    ELSE 0 
  END AS Planet_REG,
  CASE 
    WHEN d.MOBILE IS NOT NULL THEN 1 
    ELSE 0 
  END AS Existing_Customer
FROM (
  SELECT DISTINCT USER_ID, LOGIN_DATE, planet_phone_no
  FROM loc_with_mobile
) l
LEFT JOIN user_mapping u
  ON l.USER_ID = u.USER_ID
LEFT JOIN dim_agg d
  ON l.planet_phone_no = d.MOBILE
ORDER BY l.LOGIN_DATE, l.USER_ID;


Query is running:   0%|          |

Downloading:   0%|          |

In [6]:
loc_with_mobile_flagged.sample(3)

,USER_ID,LOGIN_DATE,planet_phone_no,Planet_REG,Existing_Customer
3029261,user20221217153735427,2025-09-14,9851789646,1,1
5091761,user20230113095713025,2025-09-29,8273421896,1,1
3176835,user20230712174510390,2025-09-15,9353440134,1,1


In [9]:
loc_with_mobile_flagged.Planet_REG.value_counts()

Planet_REG
1    5366203
0    328    
Name: count, dtype: Int64

In [11]:
pd.pivot_table(loc_with_mobile_flagged,index='LOGIN_DATE',aggfunc={'planet_phone_no':'nunique','Existing_Customer':'count'})

,Existing_Customer,planet_phone_no
LOGIN_DATE,,
2025-09-01,214120,85882
2025-09-02,293326,116631
2025-09-03,521123,207272
2025-09-04,304892,120845
2025-09-05,240942,96409
2025-09-06,201270,79468
2025-09-07,167045,65378
2025-09-08,218382,85087
2025-09-09,192972,75443


In [1]:
import pandas as pd
ss = pd.read_csv('gs://ltfs-analytics-prod-bucket/Analytics_Ops/Marketing Analytics/PDM/Output/general_December25_lot1.csv')
ss.shape,ss.Agreement_no.

/var/tmp/ipykernel_78562/4034061396.py:2: DtypeWarning: Columns (1,2,4,6,21,24,36,38,40,41,42,43,50) have mixed types. Specify dtype option on import or set low_memory=False.
  ss = pd.read_csv('gs://ltfs-analytics-prod-bucket/Analytics_Ops/Marketing Analytics/PDM/Output/general_December25_lot1.csv')


(33976, 65)

In [2]:
ss.sample(3)

,Agreement_no,Allocation_Date,Asset_Make,Bounce_Count,Bounce_Type,Bounce_Charge_Due,Bounce_Date,Bounce_Last_1_Month,Branchname,Bureau_Flag,...,EMI,STATE_y,LOANSTATUS,SUB_SBU,DISBURSALDATE,DOB,base_date,age,Reg_Date,segments
23835,C022062401030416,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PATIALA-CHOTI BARADARI-93,crif,...,6668.0,PUNJAB,A,CL,2022-06-25,1990-03-14,2025-11-26,35,2023-01-26,General
25972,C02511154059485377,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SIKAR-ROADWAYS BUS DEPO-69,crif,...,24467.0,RAJASTHAN,A,CL,2025-11-18,1988-09-28,2025-11-26,37,NaN,General
31196,C02405140895430340,NaN,NaN,NaN,NaN,NaN,NaN,0.0,BANGALORE-LALBAGH ROAD-LTF-13,crif,...,7479.0,KARNATAKA,A,CL,2024-05-14,1975-01-17,2025-11-26,50,2023-04-12,General


In [12]:
%%bigquery non_login_base
WITH
PLANET_REG AS (
    SELECT 
        PHONE_NO AS MOBILE, 
        USER_CREATION_ID AS USER_ID, 
        REGISTRATION_DATE AS Planet_Reg_Date
    FROM `dataeng-datalake-prod.D2C.D2C_USER_REG_LOGIN`

    UNION ALL

    SELECT 
        USER_PHNO1 AS MOBILE, 
        USER_ID,
        CREATE_TS AS Planet_Reg_Date
    FROM `edw-prod.DWH_MART.D2C_TB_ASMI_USER`
),

loc_with_mobile AS (
    SELECT 
        l.USER_ID,
        l.CREATED_TS,
        m.MOBILE AS MOBILE
    FROM (
        SELECT 
            USER_ID, 
            CREATED_TS
        FROM `edw-prod.DWH_MART.D2C_TB_USER_LOC_TRACKING`
        WHERE CREATED_TS >= '2025-11-01'
          AND CREATED_TS < '2025-11-27'

        UNION ALL

        SELECT 
            USER_CREATION_ID AS USER_ID, 
            LOGIN_DATE AS CREATED_TS
        FROM `dataeng-datalake-prod.D2C.D2C_USER_LOGIN_TRACKER`
        WHERE LOGIN_DATE >= '2025-11-01'
          AND LOGIN_DATE < '2025-11-27'
    ) l
    LEFT JOIN PLANET_REG m
        ON l.USER_ID = m.USER_ID
),

final_base AS (
    SELECT 
        pr.*,
        CASE WHEN lw.MOBILE IS NOT NULL THEN 1 ELSE 0 END AS Login_Flag,
        lw.CREATED_TS AS Last_Login_Time
    FROM PLANET_REG pr
    LEFT JOIN loc_with_mobile lw
        ON pr.MOBILE = lw.MOBILE
),

-- Deduplication: Keep latest record per MOBILE
ranked AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY MOBILE ORDER BY Last_Login_Time DESC NULLS LAST) AS rn
    FROM final_base
)

SELECT *
FROM ranked
WHERE rn = 1        -- keep latest record only
ORDER BY Last_Login_Time DESC;

Query is running:   0%|          |

Downloading:   0%|          |

In [13]:
non_login_base.shape

(12547297, 6)

In [14]:
non_login_base.shape,non_login_base.MOBILE.nunique()

((12547297, 6), 12547296)

In [15]:
non_login_base.sample(2)

,MOBILE,USER_ID,Planet_Reg_Date,Login_Flag,Last_Login_Time,rn
6571018,7381387560,user20221116214730332,2022-11-16 21:48:22.867,0,NaT,1
509755,6385730410,user20240715225112700,2024-07-15 22:52:44.410,1,2025-11-13 11:59:21.467,1


In [16]:
non_login_base.Login_Flag.value_counts()

Login_Flag
0    11485739
1     1061558
Name: count, dtype: Int64

## Disbursal Date

In [1]:
%%bigquery planet_portfolio_penetration

WITH DATE_FILTER AS (
  SELECT 
    DATE('2025-12-01') AS START_DATE,   -- 🔴 Change Start Date Here
    DATE('2026-02-28') AS END_DATE      -- 🔴 Change End Date Here
),

DIM_AGG AS (
    SELECT 
        b.AGREEMENTNO,
        a.CUSTOMERID,
        a.DOB,
        a.MOBILE,
        a.CITY,
        b.SOURCESYSTEMID,
        b.LOANSTATUS,
        b.STATE,
        b.DISBURSALDATE,
        b.DISBURSALAMOUNT,
        b.SCHEMENAME,
        b.PRODUCTNAME,
        a.UCID,
        b.PRODUCTID,
        b.NPASTAGEID
    FROM `edw-prod.DWH.LMS_DIMCUSTOMER` a
    LEFT JOIN (
        SELECT AGREEMENTNO, CUSTOMERID, SOURCESYSTEMID, LOANSTATUS, DISBURSALDATE, 
               DISBURSALAMOUNT, SCHEMENAME, PRODUCTNAME, PRODUCTID, STATE, NPASTAGEID 
        FROM `edw-prod.DWH.LMSFR_DIMAGREEMENT`
        UNION ALL
        SELECT AGREEMENTNO, CUSTOMERID, SOURCESYSTEMID, LOANSTATUS, DISBURSALDATE, 
               DISBURSALAMOUNT, SCHEMENAME, PRODUCTNAME, PRODUCTID, STATE, NPASTAGEID 
        FROM `edw-prod.DWH.LMS_DIMAGREEMENT`
        UNION ALL
        SELECT AGREEMENTNO, CUSTOMERID, SOURCESYSTEMID, LOANSTATUS, DISBURSALDATE, 
               DISBURSALAMOUNT, SCHEMENAME, PRODUCTNAME, PRODUCTID, STATE, NPASTAGEID 
        FROM `edw-prod.DWH.LMSML_DIMAGREEMENT`
    ) b
    ON a.CUSTOMERID = b.CUSTOMERID
    CROSS JOIN DATE_FILTER d
    WHERE b.AGREEMENTNO IS NOT NULL
      AND b.LOANSTATUS IN ('A', 'C', 'ACTIVE', 'CLOSED')
      AND DATE(b.DISBURSALDATE) BETWEEN d.START_DATE AND d.END_DATE
),

DIM_AGG_FINAL AS (
    SELECT *,
        CASE 
            WHEN SOURCESYSTEMID = 21 THEN "TW"
            WHEN SOURCESYSTEMID = 20 THEN "HL"
            WHEN SOURCESYSTEMID = 23 THEN "CL"
            WHEN SOURCESYSTEMID = 25 THEN "SME"
            WHEN SOURCESYSTEMID = 24 THEN "Farm"
            WHEN SOURCESYSTEMID = 27 THEN "WRF"
            WHEN SOURCESYSTEMID = 15 THEN "ML"
            WHEN SOURCESYSTEMID = 1 THEN "Farm"
            WHEN SOURCESYSTEMID = 6 THEN "Wholesale"
            WHEN SOURCESYSTEMID = 26 THEN "Rural-LAP"
            ELSE "Non-LTF"
        END AS PRODUCT,
        CASE 
            WHEN SCHEMENAME = "SUFIN" THEN "Supply-Chain-Finance"
            WHEN LOWER(PRODUCTNAME) LIKE '%lap%' 
                 OR LOWER(PRODUCTNAME) LIKE '%loan against property%' 
                    AND SOURCESYSTEMID = 20 THEN "Loan-Against-Property"
            ELSE 
                CASE 
                    WHEN SOURCESYSTEMID = 21 THEN "TW"
                    WHEN SOURCESYSTEMID = 20 THEN "HL"
                    WHEN SOURCESYSTEMID = 23 THEN "CL"
                    WHEN SOURCESYSTEMID = 25 THEN "SME"
                    WHEN SOURCESYSTEMID = 24 THEN "Farm"
                    WHEN SOURCESYSTEMID = 27 THEN "WRF"
                    WHEN SOURCESYSTEMID = 15 THEN "ML"
                    WHEN SOURCESYSTEMID = 1 THEN "Farm"
                    WHEN SOURCESYSTEMID = 6 THEN "Wholesale"
                    WHEN SOURCESYSTEMID = 26 THEN "Rural-LAP"
                    ELSE "Non-LTF"
                END
        END AS Product_Final
    FROM DIM_AGG
),

DIM_AGG_CHECK_FINAL AS (
    SELECT * FROM (
        SELECT *, 
               ROW_NUMBER() OVER (PARTITION BY UCID, Product_Final 
                                  ORDER BY DISBURSALDATE DESC) AS rn
        FROM DIM_AGG_FINAL
        WHERE UCID IS NOT NULL 
          AND MOBILE IS NOT NULL 
          AND MOBILE != 'None'
    ) 
    WHERE rn = 1
),

PRODUCT_DIST_FINAL AS (
    SELECT 
        FORMAT_DATE('%Y-%m', DATE(DISBURSALDATE)) AS DISB_MONTH,
        Product_Final,
        COUNT(DISTINCT UCID) AS CUSTOMER_COUNT,
        'PRODUCT' AS Source_Tag  
    FROM DIM_AGG_CHECK_FINAL
    GROUP BY DISB_MONTH, Product_Final
),

PLANET_REG AS (
    SELECT PHONE_NO AS MOBILE, USER_CREATION_ID AS USER_ID, 
           REGISTRATION_DATE AS Planet_Reg_Date
    FROM `dataeng-datalake-prod.D2C.D2C_USER_REG_LOGIN`
    UNION ALL
    SELECT USER_PHNO1 AS MOBILE, USER_ID, 
           CREATE_TS AS Planet_Reg_Date
    FROM `edw-prod.DWH_MART.D2C_TB_ASMI_USER`
),

PLANET_AGG AS (
    SELECT 
        p.MOBILE AS MOBILE_ASMI, 
        p.USER_ID, 
        p.Planet_Reg_Date, 
        d.AGREEMENTNO, 
        d.MOBILE, 
        d.UCID, 
        d.STATE, 
        d.Product_Final, 
        d.DISBURSALDATE
    FROM PLANET_REG p
    INNER JOIN DIM_AGG_CHECK_FINAL d
        ON p.MOBILE = d.MOBILE
),

PLANET_DIST_FINAL AS (
    SELECT 
        FORMAT_DATE('%Y-%m', DATE(DISBURSALDATE)) AS DISB_MONTH,
        Product_Final,
        COUNT(DISTINCT UCID) AS CUSTOMER_COUNT,
        'PLANET' AS Source_Tag  
    FROM PLANET_AGG
    GROUP BY DISB_MONTH, Product_Final
)

SELECT * FROM PRODUCT_DIST_FINAL
UNION ALL
SELECT * FROM PLANET_DIST_FINAL
ORDER BY DISB_MONTH, Product_Final, Source_Tag;

Query is running:   0%|          |

Downloading:   0%|          |

In [2]:
planet_portfolio_penetration.head()

,DISB_MONTH,Product_Final,CUSTOMER_COUNT,Source_Tag
0,2025-12,CL,20580,PLANET
1,2025-12,CL,40778,PRODUCT
2,2025-12,Farm,2680,PLANET
3,2025-12,Farm,10923,PRODUCT
4,2025-12,HL,844,PLANET


In [5]:
import pandas as pd
pd.pivot(planet_portfolio_penetration, index='DISB_MONTH',columns=['Product_Final','Source_Tag'],values='CUSTOMER_COUNT').fillna(0)

Product_Final     CL           Farm             HL          \
Source_Tag    PLANET PRODUCT PLANET PRODUCT PLANET PRODUCT   
DISB_MONTH                                                   
2025-12        20580   40778   2680   10923    844    1263   
2026-01        20575   43845   2589   11246    393     498   
2026-02        16524   42143   2160   10164    570    1017   

Product_Final Loan-Against-Property             ML         Non-LTF          \
Source_Tag                   PLANET PRODUCT PLANET PRODUCT  PLANET PRODUCT   
DISB_MONTH                                                                   
2025-12                         260     401  32833  270187       5      12   
2026-01                         144     181  30617  262798       2      10   
2026-02                         138     315  28652  267992       0       4   

Product_Final Rural-LAP            SME             TW            WRF          
Source_Tag       PLANET PRODUCT PLANET PRODUCT PLANET PRODUCT PLANET PRODUCT  
DISB_MONTH                                                                    
2025-12             252    1127   1203    1861  64699   64795     22      37  
2026-01             232    1219   1092    1735  83279   84083     26      44  
2026-02             239    1268    928    1769  70102   73861     47      92

## Latest Disbursal Planet campaign Base

In [25]:
%%bigquery customer_campaign_base



WITH user_mapping AS (
  SELECT 
      PHONE_NO AS planet_phone_no, 
      USER_CREATION_ID AS USER_ID,
      DATE(REGISTRATION_DATE) AS Planet_Reg_date
  FROM `dataeng-datalake-prod.D2C.D2C_USER_REG_LOGIN`

  UNION ALL

  SELECT 
      USER_PHNO1 AS planet_phone_no, 
      USER_ID,
      DATE(CREATE_TS) AS Planet_Reg_date
  FROM `edw-prod.DWH_MART.D2C_TB_ASMI_USER`
),

latest_login AS (
  SELECT 
      l.USER_ID,
      MAX(l.CREATED_TS) AS latest_login_date
  FROM (
      SELECT USER_ID, CREATED_TS
      FROM `edw-prod.DWH_MART.D2C_TB_USER_LOC_TRACKING`

      UNION ALL

      SELECT USER_CREATION_ID AS USER_ID, LOGIN_DATE AS CREATED_TS
      FROM `dataeng-datalake-prod.D2C.D2C_USER_LOGIN_TRACKER`
  ) l
  GROUP BY l.USER_ID
),

user_login_mobile AS (
  SELECT 
      m.planet_phone_no,
      m.USER_ID,
      m.Planet_Reg_date,
      ll.latest_login_date
  FROM user_mapping m
  LEFT JOIN latest_login ll
      ON m.USER_ID = ll.USER_ID
),

dim_agg AS (
  SELECT 
      b.AGREEMENTNO,
      a.MOBILE,
      b.DISBURSALDATE,
      b.PRODUCTNAME,
      b.SOURCESYSTEMID
  FROM `edw-prod.DWH.LMS_DIMCUSTOMER` a
  LEFT JOIN `edw-prod.DWH.LMSFR_DIMAGREEMENT` b
      ON a.CUSTOMERID = b.CUSTOMERID
  WHERE DATE(b.DISBURSALDATE) >= DATE_SUB(CURRENT_DATE(), INTERVAL 7 DAY) AND b.PRODUCTNAME NOT LIKE '%CL_Large_Partnership%'
),

product_logic AS (
  SELECT *,
      CASE 
        WHEN SOURCESYSTEMID = 21 THEN "TW"
        WHEN SOURCESYSTEMID = 20 THEN "HL"
        WHEN SOURCESYSTEMID = 23 THEN "CL"
        WHEN SOURCESYSTEMID = 25 THEN "SME"
        WHEN SOURCESYSTEMID = 24 THEN "Farm"
        WHEN SOURCESYSTEMID = 27 THEN "WRF"
        WHEN SOURCESYSTEMID = 15 THEN "ML"
        WHEN SOURCESYSTEMID = 1 THEN "Farm"
        WHEN SOURCESYSTEMID = 6 THEN "Wholesale"
        WHEN SOURCESYSTEMID = 26 THEN "Rural-LAP"
        ELSE "Non-LTF"
      END AS Product_Final
  FROM dim_agg
),

final_base AS (
  SELECT
      d.AGREEMENTNO,
      d.MOBILE,
      DATE(d.DISBURSALDATE) AS Disbursal_date,
      d.PRODUCTNAME,
      d.Product_Final,

      CASE 
          WHEN u.planet_phone_no IS NOT NULL THEN 1
          ELSE 0
      END AS Planet_Reg_flag,

      u.Planet_Reg_date,
      u.latest_login_date,

      DATE_DIFF(DATE(d.DISBURSALDATE), DATE(u.latest_login_date), DAY) AS Recency

  FROM product_logic d
  LEFT JOIN user_login_mobile u
      ON d.MOBILE = u.planet_phone_no
)

SELECT *
FROM (
  SELECT *,
         ROW_NUMBER() OVER(PARTITION BY AGREEMENTNO ORDER BY Disbursal_date DESC) AS rn
  FROM final_base
)
WHERE rn = 1

Query is running:   0%|          |

Downloading:   0%|          |

In [26]:
customer_campaign_base.sample(10)

,AGREEMENTNO,MOBILE,Disbursal_date,PRODUCTNAME,Product_Final,Planet_Reg_flag,Planet_Reg_date,latest_login_date,Recency,rn
5021,C02603031514621816,8093520590,2026-03-03,CONSUMER_LOAN,CL,1,2022-06-09,2026-02-18 14:30:57.260,13,1
680,C02603074911966680,9986653505,2026-03-07,CONSUMER_LOAN,CL,0,NaT,NaT,<NA>,1
5380,C02603052121491225,8146503257,2026-03-06,CONSUMER_LOAN,CL,1,2026-03-05,2026-03-08 10:49:12.483,-2,1
2546,C02602181342393211,9051808740,2026-03-04,CONSUMER_LOAN,CL,1,2025-08-23,2026-03-04 16:44:16.850,0,1
1598,C02603085877550529,9687032370,2026-03-08,CONSUMER_LOAN,CL,1,2023-01-06,2026-03-07 19:28:21.373,1,1
5340,F0059K610703260362,8374593089,2026-03-07,FARM-Kisan Gaurav,Farm,0,NaT,NaT,<NA>,1
5205,C02603045891407535,9679881881,2026-03-04,CONSUMER_LOAN,CL,1,2023-09-21,2023-09-21 10:37:48.000,895,1
1213,C02603020133638456,9974879185,2026-03-02,CONSUMER_LOAN,CL,1,2025-08-26,2025-08-26 19:04:36.577,188,1
4202,C02603031672488855,8001676923,2026-03-06,CONSUMER_LOAN,CL,1,2022-11-06,2026-03-06 13:03:27.177,0,1
4951,C02603085922162080,8609382003,2026-03-08,CONSUMER_LOAN,CL,1,2023-01-07,2026-03-08 13:46:40.680,0,1


In [27]:
customer_campaign_base.PRODUCTNAME.value_counts()

PRODUCTNAME
CONSUMER_LOAN        4303
FARM-Kisan Gaurav     956
BUSINESS_LOAN          83
FARMFIN_WRF            65
Name: count, dtype: int64

In [28]:
customer_campaign_base.shape,customer_campaign_base.AGREEMENTNO.nunique()

((5407, 10), 5407)

In [29]:
customer_campaign_base[customer_campaign_base['AGREEMENTNO']=='C02603082895327174']

,AGREEMENTNO,MOBILE,Disbursal_date,PRODUCTNAME,Product_Final,Planet_Reg_flag,Planet_Reg_date,latest_login_date,Recency,rn
88,C02603082895327174,9732000396,2026-03-08,CONSUMER_LOAN,CL,1,2024-03-22,2026-01-30 16:21:38.273,37,1


In [30]:
customer_campaign_base.Disbursal_date.min(), customer_campaign_base.Disbursal_date.max()

(datetime.date(2026, 3, 2), datetime.date(2026, 3, 8))

In [31]:
customer_campaign_base.to_csv('Newly_disbursed_customer.csv', index=False)